# পাঠ ১৩ - এজেন্ট মেমরি


## Setup

This notebook demonstrates how to build a travel booking agent with **persistent memory** using the **Microsoft Agent Framework** (MAF).

You will learn how different types of agent memory — working, short-term, and long-term — affect how an agent retains and uses information across conversations.

**Prerequisites:**
- A Microsoft Foundry project with a deployed chat model (e.g. `gpt-4.1-mini`).
- Logged in with the Azure CLI — run `az login` in your terminal.
- `AZURE_AI_PROJECT_ENDPOINT` — your Microsoft Foundry project endpoint.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — the name of your deployed model.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## এজেন্ট স্মৃতির প্রকারভেদ

AI এজেন্টগুলি বিভিন্ন ধরণের স্মৃতি ব্যবহার করতে পারে, প্রতিটি আলাদা উদ্দেশ্যে কাজ করে:

### ওয়ার্কিং মেমোরি
কথোপকথনের থ্রেড নিজেই — একটি সেশনে বিনিময় করা বার্তাসমূহ। এজেন্ট একই থ্রেডে আগের বার্তাগুলোর দিকে ফিরে দেখে সামঞ্জস্য বজায় রাখতে পারে। MAF এ এটি তৈরি হয় **`agent.create_session()`** মাধ্যমে, যা একটি `AgentSession` প্রদান করে।

### সংক্ষিপ্ত মেয়াদী স্মৃতি
এমন তথ্য যা একটি কাজ বা সেশনের সময়কাল পর্যন্ত টিকে থাকে কিন্তু স্থায়ীভাবে সংরক্ষিত হয় না। উদাহরণস্বরূপ, একটি মাল্টি-টার্ন পরিকল্পনা কথোপকথনের সময় এজেন্ট কিছু তথ্য সংগ্রহ করতে পারে এবং সেগুলো ব্যবহার করে একটি চূড়ান্ত ভ্রমণসূচী তৈরি করতে পারে।

### দীর্ঘমেয়াদী স্মৃতি
পছন্দ এবং তথ্য যা **সেশনজুড়ে** টিকে থাকে। ফিরে আসা ব্যবহারকারীকে তাদের খাদ্য সীমাবদ্ধতা বা ভ্রমণের ধরন বারবার বলা লাগবে না। দীর্ঘমেয়াদী স্মৃতির পেছনে সাধারণত একটি বাইরের সংরক্ষণাগার থাকে — একটি ডাটাবেস, ফাইল, বা ভেক্টর ইনডেক্স — এবং এজেন্টের কাছে সরঞ্জামের মাধ্যমে উপস্থাপন করা হয়।


## সেশন সহ ওয়ার্কিং মেমোরি

মেমোরির সবচেয়ে সহজ রূপ হল কথোপকথনের সেশন। যখন আপনি একই সেশনটি (যা `agent.create_session()` মাধ্যমে তৈরি হয়) ধারাবাহিক `agent.run()` কলগুলিতে পাঠান, তখন এজেন্ট সেই কথোপকথনের সম্পূর্ণ ইতিহাস দেখে এবং পূর্বের বিবরণ স্মরণ করতে পারে।

চলুন একটি ট্রাভেল এজেন্ট তৈরি করি এবং ওয়ার্কিং মেমোরি প্রদর্শন করি।


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

এজেন্টটি সঠিকভাবে বাজেটটি স্মরণ করেছিল কারণ উভয় বার্তা একই সেশন শেয়ার করে। এটিই **কর্মক্ষম স্মৃতি** — এটি শুধুমাত্র সেশনের জীবনের জন্য বিদ্যমান।

### নতুন থ্রেডের সাথে কি ঘটে?

যদি আমরা একটি **নতুন** সেশন তৈরি করি, তাহলে এজেন্টের পূর্ববর্তী কথোপকথনের কোনো স্মৃতি থাকে না:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## দীর্ঘমেয়াদী মেমোরি প্যাটার্ন

ব্যবহারকারীর পছন্দ **সেশন জুড়ে** মনে রাখার জন্য, আমাদের এমন একটি স্থায়ী সংরক্ষণ প্রয়োজন যা কথোপকথনের থ্রেডের বাইরে থাকে। এজেন্ট এই সংরক্ষণ থেকে তথ্য সংরক্ষণ এবং পুনরুদ্ধারের জন্য **টুলস** — ফাংশনগুলি কল করে — অ্যাক্সেস করে।

নীচে আমরা একটি সহজ ইন-মেমোরি পছন্দ সংরক্ষণ প্রয়োগ করেছি (প্রোডাকশনে আপনি এর পিছনে একটি ডাটাবেস বা ভেক্টর ইনডেক্স ব্যবহার করবেন) এবং এটি টুলস হিসাবে প্রকাশ করেছি যা এজেন্ট ব্যবহার করতে পারে।

### আর্কিটেকচার
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### দৃশ্য ১ — প্রথমবারের ব্যবহারকারী বার্ষিকীর যাত্রা বুক করে

সারাহ প্রথমবার আসেন। এজেন্টকে তার পছন্দগুলি টুলগুলির মাধ্যমে সংরক্ষণ করতে হবে এবং হোটেলগুলি সুপারিশ করতে সেগুলি ব্যবহার করতে হবে।


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### দৃশ্য ২ — সারা কয়েক সপ্তাহ পরে ফিরে আসছেন

সারা একটি **সম্পূর্ণ নতুন থ্রেড** শুরু করেন (নতুন সেশন অনুকরণ করছে)। ওয়ার্কিং মেমরি খালি, কিন্তু দীর্ঘমেয়াদী প্রেফারেন্স স্টোরে তার তথ্য এখনও আছে। এজেন্টকে এটি উদ্ধার করে ব্যক্তিগতকৃত সুপারিশের জন্য ব্যবহার করা উচিত।


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## সারাংশ

এই পাঠে আপনি তিন ধরনের এজেন্ট মেমরি এবং কীভাবে সেগুলি Microsoft Agent Framework দিয়ে বাস্তবায়ন করতে হয় তা শিখেছেন:

| মেমরি প্রকার | MAF প্রক্রিয়া | আয়ুকাল |
|---|---|---|
| **কর্মরত** | `agent.create_session()` | একক সংলাপ |
| **স্বল্পমেয়াদী** | থ্রেডের মধ্যে সঞ্চিত প্রেক্ষাপট | একক কাজ / সেশন |
| **দীর্ঘমেয়াদী** | `@tool` ফাংশনগুলির মাধ্যমে বাহ্যিক সংরক্ষণাগারে অ্যাক্সেস | সেশনের বাইরে |

### মূল অংশগুলি
1. **`agent.create_session()`** প্রদান করে কর্মরত মেমরি — এজেন্ট একটি সেশনের মধ্যে সম্পূর্ণ সংলাপ ইতিহাস দেখতে পারে।
2. **নতুন সেশনগুলো প্রেক্ষাপট হারায়** — দীর্ঘমেয়াদী মেমরি ছাড়া এজেন্ট পূর্ব সংলাপ স্মরণ করতে পারে না।
3. **`@tool` ফাংশনগুলো** ফাঁক পূরণ করে — এজেন্টকে একটি স্থায়ী সংরক্ষণাগার থেকে তথ্য সঞ্চয় এবং পুনরুদ্ধার করার সুযোগ দেয়।
4. **ব্যক্তিগতকরণ সময়ের সঙ্গে উন্নত হয়** — যত বেশি পছন্দ সংরক্ষিত হয়, এজেন্টের প্রস্তাবনা তত উন্নত হয়।

### বাস্তব জীবনের প্রয়োগসমূহ
- **কাস্টমার সার্ভিস**: গ্রাহকের ইতিহাস এবং পছন্দগুলি মনে রাখা
- **ব্যক্তিগত সহকারী**: দিন বা সপ্তাহ পরিপ্রেক্ষিত ধরে রাখা
- **স্বাস্থ্যসেবা**: রোগীর তথ্য এবং পছন্দ অনুসরণ করা
- **ই-কমার্স**: ইতিহাসের ভিত্তিতে ব্যক্তিগতকৃত কেনাকাটা

### পরবর্তী ধাপসমূহ
- ইন-মেমরি ডিক্টের পরিবর্তে একটি ডাটাবেস বা ভেক্টর স্টোর ব্যবহার করুন (যেমন Azure AI Search)
- সময়-সংবেদনশীল তথ্যের জন্য মেমরি মেয়াদ শেষ করার ব্যবস্থা যোগ করুন
- যৌক্তিক স্মৃতি সহ মাল্টি-এজেন্ট সিস্টেম তৈরি করুন
- জ্ঞান-গ্রাফ সমর্থিত স্মৃতির জন্য Cognee নোটবুক অন্বেষণ করুন


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
